**API step — run locally with your OpenAI key.** Generates DiCE- and SHAP-grounded consultation guides for the 40 evaluation cases. Reads `data/eval_cases.parquet`; writes `results/llm/guides.jsonl`.

In [8]:
# 10_llm_guide_generation.ipynb
# API STEP -- RUN LOCALLY. Generates DiCE- and SHAP-grounded consultation guides for the 40
# evaluation cases. Reads data/eval_cases.parquet (from notebook 07); writes results/llm/guides.jsonl.
# Generation model from .env (LLM_MODEL_WEAK=gpt-4o-mini), temperature=0.3. Requires: openai>=1.0, python-dotenv.

import os, json, time
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

# resolve repo root whether this notebook runs from the repo root or from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
OUT = ROOT / "results" / "llm"; OUT.mkdir(parents=True, exist_ok=True)

load_dotenv(ROOT / ".env")                      # loads OPENAI_API_KEY and LLM_MODEL_* from .env
GEN_MODEL = os.getenv("LLM_MODEL_WEAK", "gpt-4o-mini")
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

PRED = ["age", "HE_BMI", "HE_wc", "HE_wt", "HE_ht", "N_EN", "N_CHO", "N_SUGAR",
        "N_CHOL", "sm_presnt", "dr_month", "pa_aerobic", "L_BR_FQ", "edu", "ho_incm", "D_1_1"]
LAB = {"age": "age (years)", "HE_BMI": "BMI", "HE_wc": "waist circumference (cm)",
       "HE_wt": "weight (kg)", "HE_ht": "height (cm)", "N_EN": "energy intake (kcal)",
       "N_CHO": "carbohydrate (g)", "N_SUGAR": "sugar (g)", "N_CHOL": "cholesterol (mg)",
       "sm_presnt": "current smoker", "dr_month": "monthly alcohol", "pa_aerobic": "meets aerobic guideline",
       "L_BR_FQ": "breakfast frequency code", "edu": "education level", "ho_incm": "income quartile",
       "D_1_1": "self-rated health"}

SYSTEM_PROMPT = (
    "You are an insurance risk-communication assistant. You write a short consultation guide "
    "for an applicant based on the output of a diabetes-risk classifier. Follow these rules:\n"
    "1. Frame every risk figure as a STATISTICAL ASSOCIATION from a cross-sectional model, "
    "never as a causal prediction or a promise of prevention.\n"
    "2. Explain that the model describes how similar the applicant's profile is to higher-risk "
    "profiles in the data; it does not establish that changing a feature will change their health.\n"
    "3. Use ADA/IDF numerical reference points where relevant (HbA1c >= 6.5% diabetes; "
    "5.7-6.4% prediabetes; fasting plasma glucose >= 126 mg/dL).\n"
    "4. Do NOT give a medical diagnosis or prescription.\n"
    "5. State the applicant's risk tier in actuarial language and the premium-review implication, "
    "in association-based wording only.\n"
    "6. Keep it readable for a layperson.\n"
    "7. End with a MANDATORY disclaimer: this is an internal risk analysis, not medical advice, "
    "and the applicant should consult a qualified healthcare professional.\n"
    "Return the guide using these sections: (1) Risk profile summary; (2) Explanation interpretation; "
    "(3) Association-based health-awareness points; (4) Cross-sectional data caveat; (5) Disclaimer."
)

def dice_user_prompt(r):
    changed = [c for c in r["dice_changed"].split("|") if c]
    lines = [f"- {LAB[c]}: {r['orig_'+c]} -> {r['cf_'+c]} (counterfactual target)" for c in changed]
    return (f"Applicant {r['case_id']} ({r['sex']}). Model predicted diabetes-risk probability "
            f"{r['orig_prob']:.2f} (elevated-risk tier, top of the screened distribution). A COUNTERFACTUAL "
            f"explanation (DiCE) indicates the following feature targets associated with a lower predicted "
            f"risk of {r['dice_cf_prob']:.2f}:\n" + "\n".join(lines) +
            "\nWrite the consultation guide grounded in these counterfactual targets.")

def shap_user_prompt(r):
    top = [c for c in r["shap_top3"].split("|") if c]
    lines = [f"- {LAB[c]} (current value {r['orig_'+c]})" for c in top]
    return (f"Applicant {r['case_id']} ({r['sex']}). Model predicted diabetes-risk probability "
            f"{r['orig_prob']:.2f} (elevated-risk tier). A FEATURE-ATTRIBUTION explanation (SHAP) identifies "
            f"the following features as most associated with the predicted risk (direction only, no target "
            f"value):\n" + "\n".join(lines) +
            "\nWrite the consultation guide grounded in these attributions.")

def generate(system, user):
    resp = client.chat.completions.create(
        model=GEN_MODEL, temperature=0.3,
        messages=[{"role": "system", "content": system}, {"role": "user", "content": user}])
    return resp.choices[0].message.content

cases = pd.read_parquet(DATA / "eval_cases.parquet")
out_path = OUT / "guides.jsonl"
with open(out_path, "w") as f:
    for _, r in cases.iterrows():
        for method, builder in [("dice", dice_user_prompt), ("shap", shap_user_prompt)]:
            try:
                text = generate(SYSTEM_PROMPT, builder(r))
            except Exception as e:
                text = f"[ERROR: {e}]"
            f.write(json.dumps({"case_id": r["case_id"], "sex": r["sex"],
                                "method": method, "guide": text}) + "\n")
            time.sleep(0.2)
        print(f"generated guides for {r['case_id']}")
print(f"\nSaved results/llm/guides.jsonl (2 guides x {len(cases)} cases). Next: run 11_llm_judge_evaluation.ipynb")

generated guides for M00
generated guides for M01
generated guides for M02
generated guides for M03
generated guides for M04
generated guides for M05
generated guides for M06
generated guides for M07
generated guides for M08
generated guides for M09
generated guides for M10
generated guides for M11
generated guides for M12
generated guides for M13
generated guides for M14
generated guides for M15
generated guides for M16
generated guides for M17
generated guides for M18
generated guides for M19
generated guides for F00
generated guides for F01
generated guides for F02
generated guides for F03
generated guides for F04
generated guides for F05
generated guides for F06
generated guides for F07
generated guides for F08
generated guides for F09
generated guides for F10
generated guides for F11
generated guides for F12
generated guides for F13
generated guides for F14
generated guides for F15
generated guides for F16
generated guides for F17
generated guides for F18
generated guides for F19
